# Бинарная классификация табличных данных — прогнозирование сердечных заболеваний

В этом ноутбуке обучается нейронная сеть Keras для бинарной классификации пациентов по данным `heart.csv`.

## Импорт библиотек

Используем Pandas для работы с таблицей, NumPy для массивов, TensorFlow/Keras для нейронной сети и Matplotlib для графиков.

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

## Фиксируем случайность

Это позволяет получать воспроизводимые результаты при повторном запуске ноутбука.

In [ ]:
keras.utils.set_random_seed(42)

## Загрузка данных

In [ ]:
df = pd.read_csv("heart.csv")

In [ ]:
df.shape

In [ ]:
df.head(3)

In [ ]:
df.target.value_counts(normalize=True, dropna=False)

## Предварительная обработка

Категориальные признаки преобразуем с помощью one-hot encoding, а числовые признаки стандартизируем.

Сначала разделяем данные на обучающую и тестовую выборки, чтобы статистики для нормализации вычислялись только по обучающим данным.

In [ ]:
categorical_variables = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal']
numerics = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'slope']

In [ ]:
df1 = pd.get_dummies(df, columns=categorical_variables, dtype=int)

In [ ]:
df1.head(3)

In [ ]:
test_df = df1.sample(frac=0.2, random_state=42)
train_df = df1.drop(test_df.index)

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
means = train_df[numerics].mean()
sd = train_df[numerics].std()

In [ ]:
means

In [ ]:
train_df[numerics] = (train_df[numerics] - means) / sd
test_df[numerics] = (test_df[numerics] - means) / sd

In [ ]:
train_df.head()

Теперь все признаки являются числовыми, поэтому преобразуем датафреймы в массивы NumPy.

In [ ]:
train = train_df.to_numpy()
test = test_df.to_numpy()

Целевая переменная `target` находится в шестом столбце (индекс 6). Отделяем её от признаков.

In [ ]:
train_X = np.delete(train, 6, axis=1)
test_X = np.delete(test, 6, axis=1)

In [ ]:
train_X.shape, test_X.shape

In [ ]:
train_y = train[:, 6]
test_y = test[:, 6]

In [ ]:
train_y.shape, test_y.shape

## Создание модели

Используем один скрытый слой из 16 нейронов с ReLU и один выходной нейрон с sigmoid. Это задача бинарной классификации.

In [ ]:
num_columns = train_X.shape[1]

input = keras.Input(shape=(num_columns,))
h = keras.layers.Dense(16, activation="relu", name="Hidden")(input)
output = keras.layers.Dense(1, activation="sigmoid", name="Output")(h)

model = keras.Model(input, output)

In [ ]:
model.summary()

In [ ]:
(29 + 1) * 16 + (16 + 1) * 1

In [ ]:
keras.utils.plot_model(model, show_shapes=True)

## Компиляция модели

Для бинарной классификации используем `binary_crossentropy`, оптимизатор Adam и метрику accuracy.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## Обучение модели

Обучаем модель 300 эпох с размером пакета 32. 20% обучающих данных автоматически используются для валидации.

In [ ]:
history = model.fit(
    train_X,
    train_y,
    epochs=300,
    batch_size=32,
    verbose=1,
    validation_split=0.2
)

## Анализ обучения

Сохраняем историю обучения и строим графики ошибки и точности.

In [ ]:
history_dict = history.history
history_dict.keys()

In [ ]:
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)

plt.plot(epochs, loss_values, "bo", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.clf()

acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]

plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Оценка модели

Проверяем качество модели на тестовой выборке, которая не использовалась при обучении.

In [ ]:
model.evaluate(test_X, test_y)